In [2]:
import os
from dotenv import load_dotenv, find_dotenv 

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"] = "Agents"

In [3]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
embeddings = OpenAIEmbeddings() 
model = ChatOpenAI()

#### **Simple AI Assistant**

In [5]:
# while True: 
#     question = input("Enter the question: ")
#     if question != "quit":
#         print(model.invoke(question).content) 
#     else:
#         print("Good-bye take care")
#         break

In [6]:
from langchain_core.chat_history import BaseChatMessageHistory 
from langchain_core.chat_history import InMemoryChatMessageHistory 
from langchain_core.runnables.history import RunnableWithMessageHistory 
from langchain_core.messages import AIMessage 

In [7]:
store = {}

In [8]:
def get_session_history(session_id : str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory() 
    return store[session_id]

In [9]:
config = {"configurable": {"session_id": "firstchat"}}

In [10]:
model_with_memory = RunnableWithMessageHistory(model, get_session_history)

In [11]:
model_with_memory.invoke(("Hi my name is Hitesh Ram, I love Artificial Intelligence"), config=config).content

"Hello Hitesh Ram! It's great to hear that you love Artificial Intelligence. It's a fascinating field with so much potential for innovation and advancement. What specifically about AI interests you the most?"

In [16]:
model_with_memory.invoke(("Whats my name?"), config=config).content

'Your name is Hitesh Ram.'

In [17]:
model_with_memory.invoke(("What i love doing ?"), config=config).content

'You mentioned that you love Artificial Intelligence. It seems like you have a passion for technology and innovation in the field of AI. Is there anything specific within AI that you enjoy working on or learning about?'

In [18]:
store

{'firstchat': InMemoryChatMessageHistory(messages=[HumanMessage(content='Hi my name is Hitesh Ram, I love Artificial Intelligence', additional_kwargs={}, response_metadata={}), AIMessage(content="Hello Hitesh Ram! It's great to hear that you love Artificial Intelligence. It's a fascinating field with so much potential for innovation and advancement. What specifically about AI interests you the most?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 20, 'total_tokens': 60, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-80a5bda8-8bb8-4270-a347-277de66099b4-0', usage_metadata={'input_tokens': 20, 'output_tokens': 40, 'total_tokens': 60, 'input_token_deta

### **Tools & Agents**

In [19]:
from langchain_community.tools import WikipediaQueryRun 
from langchain_community.utilities import WikipediaAPIWrapper

In [20]:
api_wrapper = WikipediaAPIWrapper()

In [21]:
tool = WikipediaQueryRun(api_wrapper=api_wrapper)

In [22]:
tool.name

'wikipedia'

In [23]:
tool.description

'A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.'

In [24]:
tool.args

{'query': {'description': 'query to look up on wikipedia',
  'title': 'Query',
  'type': 'string'}}

In [25]:
print(tool.run({"query" : "langchain"}))

Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.

Page: Retrieval-augmented generation
Summary: Retrieval Augmented Generation (RAG) is a technique that grants generative artificial intelligence models information retrieval capabilities. It modifies interactions with a large language model (LLM) so that the model responds to user queries with reference to a specified set of documents, using this information to augment information drawn from its own vast, static training data. This allows LLMs to use domain-specific and/or updated information.  
Use cases include providing chatbot access to internal company data or giving factual information only from an authoritative source.

Page: Sentence em

In [27]:
from langchain_community.tools import YouTubeSearchTool

In [28]:
tool2 = YouTubeSearchTool()

In [30]:
tool2.name

'youtube_search'

In [35]:
tool2.args

{'query': {'title': 'Query', 'type': 'string'}}

In [37]:
tool2.run({"query": "Joe Rogan"})

"['https://www.youtube.com/watch?v=7qZl_5xHoBw&pp=ygUJSm9lIFJvZ2Fu', 'https://www.youtube.com/watch?v=_PcEmlVdgeo&pp=ygUJSm9lIFJvZ2Fu']"

In [38]:
from langchain_community.tools.tavily_search import TavilySearchResults

In [39]:
tool3 = TavilySearchResults()

In [40]:
tool3.invoke({"query": "Who won 2024 election in USA"})

[{'url': 'https://www.reuters.com/graphics/USA-ELECTION/RESULTS/zjpqnemxwvx/',
  'content': 'Table showing the percent of votes for the top Democratic and Republican candidates and the percent of expected votes reported for each state in president races that are Solid Democrat. Table showing the percent of votes for the top Democratic and Republican candidates and the percent of expected votes reported for each state in president races that are Competitive. Table showing the percent of votes for the top Democratic and Republican candidates and the percent of expected votes reported for each state in president races that are Lean Republican. Table showing the percent of votes for the top Democratic and Republican candidates and the percent of expected votes reported for each state in president races that are Solid Republican.'},
 {'url': 'https://www.nbcnews.com/politics/2024-elections/president-results',
  'content': "Democrats won Nevada in the past two presidential elections — but po

In [41]:
from langchain.agents import AgentType 
from langchain.agents import load_tools 
from langchain.agents import initialize_agent 

In [43]:
tool = load_tools(["wikipedia"], llm = model)

In [44]:
agent = initialize_agent(tool, llm = model, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose = True)

In [46]:
agent.run("List all the olymics medals winner from india in 2024")



> Entering new AgentExecutor chain...
I should search for a list of Olympic medal winners from India in 2024 to get the information I need.
Action: wikipedia
Action Input: List of Olympic medal winners from India in 2024
Observation: Page: List of 2024 Summer Olympics medal winners
Summary: The programme of the 2024 Summer Olympics featured 329 events in 32 sports, including the 28 "core" Olympic sports contested in 2016 and 2020, and four optional sports that were proposed by the Paris Organising Committee: breaking made its Olympic debut as an optional sport, while skateboarding, sport climbing, and surfing returned from 2020.



Page: 2024 Summer Olympics medal table
Summary: The 2024 Summer Olympics, officially known as the Games of the XXXIII Olympiad, were an international multi-sport event held in Paris, France, from 26 July to 11 August 2024, with preliminary events in some sports beginning on 24 July. Athletes representing 206 National Olympic Committees (NOCs) participated 

"The Olympic medal winners from India in 2024 were Manu Bhaker (2 bronze in shooting), Neeraj Chopra (1 silver in men's javelin throw), and Aman Sehrawat (1 bronze in wrestling)."

In [50]:
tool = TavilySearchResults()
tool.invoke({"query": "Who is the winner of wrestlemania 2024"})

[{'url': 'https://www.wwe.com/shows/wrestlemania/article/full-wrestlemania-results-2024',
  'content': "FULL WRESTLEMANIA RESULTS WrestleMania Sunday Post show press conference DAMIAN PRIEST ... SmackDown results, Nov. 15, 2024: Roman Reigns is left looking for answers after Paul Heyman's cell phone is disconnected . Full Results. WWE NXT results, Nov. 19, 2024: Chase U is no more as Ridge Holland becomes No. 1 Contender to the NXT Title"},
 {'url': 'https://www.scotsman.com/sport/other-sport/wrestlemania-results-wrestlemania-xl-2024-results-winners-list-wrestlemania-40-4582639',
  'content': 'Wrestlemania results: Wrestlemania XL 2024 results, winners list Wrestlemania 40. By Graham Falk. Sports Writer. Comment. Published 8th Apr 2024, 10:23 GMT.'},
 {'url': 'https://www.forbes.com/sites/alfredkonuwa/2024/04/06/wwe-wrestlemania-40-results-winners-and-grades-from-night-1/',
  'content': "WWE WrestleMania 40 Results on April 6, 2024 Rhea Ripley def. Becky Lynch | WWE Women's World Champ